In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, MapType
from datetime import date
from pyspark.sql.functions import *

# Assume spark is already created
spark = SparkSession.builder.appName("PracticeDataset").getOrCreate()

# Define the schema
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("tags", StringType(), True), # For split() and regex examples
    StructField("details", StringType(), True), # For translate(), substring() examples
    StructField("features", StringType(), True), # For regexp_extract(), regexp_replace() examples
    StructField("product_attributes", MapType(StringType(), StringType()), True), # For MapType, explode, map_keys, map_values
    StructField("related_products", StringType(), True) # For collect_list, collect_set
])

# Sample data
data = [
    (1, 101, "Laptop", 1, 1200.00, date(2023, 1, 15), "electronics,office", "Serial: A1B2C3D4", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"),
    (2, 102, "Mouse", 2, 25.50, date(2023, 1, 15), "electronics,accessory", "Model: M500", "user_456_mouse", {"color": "black", "wireless": "true"}, "Laptop,Monitor"),
    (3, 101, "Keyboard", 1, 75.00, date(2023, 1, 16), "electronics,accessory", "SKU: KB-101", "id_789_keyboard", {"layout": "US", "mechanical": "false"}, "Mouse,Monitor"),
    (4, 103, "Monitor", 1, 300.00, date(2023, 1, 16), "electronics,display", "DisplaySize: 27inch", "user_123_monitor", {"size": "27", "resolution": "1080p"}, "Laptop,Keyboard"),
    (5, 102, "Desk Chair", 1, 150.00, date(2023, 1, 17), "furniture,office", "Weight: 20kg", "user_456_chair", {"material": "mesh", "adjustable": "true"}, "Desk,Lamp"),
    (6, 104, "Lamp", 2, 35.00, date(2023, 1, 17), "furniture,lighting", "BulbType: LED", "id_abc_lamp", None, "Desk Chair,Desk"), # Example with None map
    (7, 101, "Laptop", 1, 1200.00, date(2023, 1, 18), "electronics,office", "Serial: E5F6G7H8", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"), # Duplicate order
    (8, 105, "Notebook", 5, 3.00, date(2023, 1, 18), "office,stationery", "Pages: 100", "user_xyz_notebook", {}, "Pen,Pencil"), # Example with empty map
    (9, 103, "Desk", 1, 250.00, date(2023, 1, 19), "furniture,office", "Material: Wood", "user_789_desk", {"size": "medium"}, "Desk Chair,Lamp"),
    (10, 104, "Pen", 10, 1.50, date(2023, 1, 19), "office,stationery", "InkColor: Blue", "id_def_pen", {"color": "blue"}, "Notebook,Pencil"),
    (11, 105, "Pencil", 12, 0.50, date(2023, 1, 20), "office,stationery", "LeadSize: 0.7mm", "user_xyz_pencil", {"lead": "0.7"}, "Notebook,Pen"),
    (12, 106, "Tablet", 1, 400.00, date(2023, 1, 20), "electronics", "Model: Tab-Pro", "user_abc_tablet", {"os": "Android"}, None), # Example with None related_products
    (13, 106, "Protector", 1, 15.00, date(2023, 1, 20), "electronics,accessory", "Type: Screen", "user_abc_protector", {}, ""), # Example with empty related_products
    # Added new rows
    (14, 101, "Mouse", 1, 25.50, date(2023, 1, 21), "electronics,accessory", "Model: M600", "user_101_mouse", {"color": "white", "wireless": "false"}, "Keyboard"),
    (15, 102, "Laptop", 1, 1100.00, date(2023, 1, 21), "electronics,office", "Serial: I9J10K11L12", "user_102_laptop", {"color": "black", "brand": "UVW"}, "Mouse,Monitor"),
    (16, 103, "Keyboard", 2, 70.00, date(2023, 1, 22), "electronics,accessory", "SKU: KB-202", "user_103_keyboard", {"layout": "UK", "mechanical": "true"}, "Mouse"),
    (17, 104, "Desk", 1, 220.00, date(2023, 1, 22), "furniture,office", "Material: Metal", "user_104_desk", {"size": "large"}, "Chair"),
    (18, 105, "Lamp", 1, 30.00, date(2023, 1, 23), "furniture,lighting", "BulbType: Incandescent", "user_105_lamp", None, "Desk"),
    (19, 106, "Notebook", 3, 2.50, date(2023, 1, 23), "office,stationery", "Pages: 150", "user_106_notebook", {}, "Pen,Pencil"),
    (20, 101, "Monitor", 1, 280.00, date(2023, 1, 24), "electronics,display", "DisplaySize: 24inch", "user_101_monitor", {"size": "24", "resolution": "1080p"}, "Laptop")
]

practice_df = spark.createDataFrame(data, schema)

# Show the DataFrame and its schema
print("Sample Practice DataFrame:")
practice_df.show(truncate=False)
practice_df.printSchema()

Sample Practice DataFrame:
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+
|order_id|customer_id|product   |quantity|price |order_date|tags                 |details               |features          |product_attributes                    |related_products      |
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+
|1       |101        |Laptop    |1       |1200.0|2023-01-15|electronics,office   |Serial: A1B2C3D4      |user_123_laptop   |{color -> silver, brand -> XYZ}       |Keyboard,Mouse,Monitor|
|2       |102        |Mouse     |2       |25.5  |2023-01-15|electronics,accessory|Model: M500           |user_456_mouse    |{color -> black, wireless -> true}    |Laptop,Monitor        |
|3       |101        |Keyboard  |1    

In [2]:
'''Extract each individual tag from the tags column and create a new row for each tag,
keeping the order_id.'''

from pyspark.sql.functions import explode, col, split
practice_df_tags = practice_df.select(col("order_id"), explode(split(col("tags"), ",")).alias("tag"))
practice_df_tags.show(truncate=False)



+--------+-----------+
|order_id|tag        |
+--------+-----------+
|1       |electronics|
|1       |office     |
|2       |electronics|
|2       |accessory  |
|3       |electronics|
|3       |accessory  |
|4       |electronics|
|4       |display    |
|5       |furniture  |
|5       |office     |
|6       |furniture  |
|6       |lighting   |
|7       |electronics|
|7       |office     |
|8       |office     |
|8       |stationery |
|9       |furniture  |
|9       |office     |
|10      |office     |
|10      |stationery |
+--------+-----------+
only showing top 20 rows



In [3]:
#Count how many orders belong to each unique tag.
from pyspark.sql.functions import count
practice_df_tags.groupby("tag").agg(count(col("order_id")).alias("order counts")).show()

+-----------+------------+
|        tag|order counts|
+-----------+------------+
|     office|          10|
|    display|           2|
| stationery|           4|
|   lighting|           2|
|  furniture|           5|
|electronics|          11|
|  accessory|           5|
+-----------+------------+



In [4]:
'''Create a new column that combines the product and quantity columns
into a single string like "Laptop (1)"'''

from pyspark.sql.functions import concat_ws, lit
practice_df.withColumn("Name&qty", concat_ws(" ", col("product"), concat_ws("",lit("("), concat_ws("", col("quantity"), lit(")"))))).show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|      Name&qty|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|Keyboard,Mouse,Mo...|    Laptop (1)|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black, ...|      Laptop,Monitor|     Mouse (2)|
|       3|        101|  Keyboard|       1|  75.0|2023-01-16|electronics,acces...|         SKU: KB-101|   

In [5]:
'''Combine the elements of the related_products
string into a list using a comma as a separator (you might need split() first).'''

practice_df_modified = practice_df.withColumn("related_products", split(col("related_products"), ","))
practice_df_relprodlist = practice_df_modified.withColumn("Related Product List", concat_ws(",", col("related_products")))
practice_df_relprodlist.show(truncate=False)

+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+--------------------------+----------------------+
|order_id|customer_id|product   |quantity|price |order_date|tags                 |details               |features          |product_attributes                    |related_products          |Related Product List  |
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+--------------------------+----------------------+
|1       |101        |Laptop    |1       |1200.0|2023-01-15|electronics,office   |Serial: A1B2C3D4      |user_123_laptop   |{color -> silver, brand -> XYZ}       |[Keyboard, Mouse, Monitor]|Keyboard,Mouse,Monitor|
|2       |102        |Mouse     |2       |25.5  |2023-01-15|electronics,accessory|Model: M500           |user_456_mouse    |{color -> black, wir

In [6]:
'''Explode the product_attributes map to have one row per attribute key-value pair,
keeping the order_id and product.'''

prod_attr_df = practice_df_relprodlist.select(col("order_id"), col("product"), explode(col("product_attributes").alias("Attributes", "features")))
prod_attr_df.show()


+--------+----------+----------+-------+
|order_id|   product|       key|  value|
+--------+----------+----------+-------+
|       1|    Laptop|     color| silver|
|       1|    Laptop|     brand|    XYZ|
|       2|     Mouse|     color|  black|
|       2|     Mouse|  wireless|   true|
|       3|  Keyboard|    layout|     US|
|       3|  Keyboard|mechanical|  false|
|       4|   Monitor|      size|     27|
|       4|   Monitor|resolution|  1080p|
|       5|Desk Chair|  material|   mesh|
|       5|Desk Chair|adjustable|   true|
|       7|    Laptop|     color| silver|
|       7|    Laptop|     brand|    XYZ|
|       9|      Desk|      size| medium|
|      10|       Pen|     color|   blue|
|      11|    Pencil|      lead|    0.7|
|      12|    Tablet|        os|Android|
|      14|     Mouse|     color|  white|
|      14|     Mouse|  wireless|  false|
|      15|    Laptop|     color|  black|
|      15|    Laptop|     brand|    UVW|
+--------+----------+----------+-------+
only showing top

In [7]:
'''Get a list of all unique attribute keys present in the
product_attributes column across all orders.'''

from pyspark.sql.functions import map_keys, col

all_keys = practice_df_relprodlist.select(map_keys(col("product_attributes")).alias("all_attributes"))

exploded_all_keys = all_keys.select(explode(col("all_attributes")).alias("exploded_attributes"))

exploded_all_keys.distinct().show()

+-------------------+
|exploded_attributes|
+-------------------+
|           material|
|           wireless|
|              brand|
|              color|
|         resolution|
|         adjustable|
|             layout|
|         mechanical|
|               size|
|                 os|
|               lead|
+-------------------+



In [10]:
#From the details column, extract the serial number (e.g., "A1B2C3D4") for products where the detail starts with "Serial: ".

from pyspark.sql.functions import regexp_extract, col
serial_df = practice_df_relprodlist.withColumn("serial_number", regexp_extract(col("details"), r"Serial: (.*)", 1))
serial_df.show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|serial_number|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|     A1B2C3D4|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black, ...|   [Laptop, Monitor]|      Laptop,Monitor|             |
|    

In [15]:
#From the features column, extract the numeric ID (e.g., "123") that appears after "user_".

features_numdf = serial_df.withColumn("Num_features", regexp_extract(col("features"), r"user_(\d+)", 1))
features_numdf.show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|serial_number|Num_features|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|     A1B2C3D4|         123|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black, ...|   [Laptop

In [17]:
#Create a new column by removing all vowels (both uppercase and lowercase) from the details column.
from pyspark.sql.functions import translate, col
serial_df.withColumn("no_vowels", translate(col("details"), "aeiouAEIOU", "")).show()


+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+----------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|serial_number|       no_vowels|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+----------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|     A1B2C3D4|    Srl: 1B2C3D4|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black

In [18]:
#Extract the first 5 characters of the features column.

from pyspark.sql.functions import substring
serial_df.withColumn("first5chars", substring(col("features"), 1, 5 )).show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+-----------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|serial_number|first5chars|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+-------------+-----------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|     A1B2C3D4|      user_|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black, ...|   [Laptop, Mo

In [21]:
#For each customer_id, create a list of all products they have ordered (collect_list).
from pyspark.sql.functions import collect_list, collect_set
prod_list_df = practice_df_relprodlist.groupby("customer_id").agg(collect_list(col("product")))
prod_list_df.show()

#For each customer_id, create a set of unique products they have ordered (collect_set).
prod_set_df = practice_df_relprodlist.groupby("customer_id").agg(collect_set(col("product")))
prod_set_df.show()

+-----------+---------------------+
|customer_id|collect_list(product)|
+-----------+---------------------+
|        101| [Laptop, Keyboard...|
|        103| [Monitor, Desk, K...|
|        102| [Mouse, Desk Chai...|
|        105| [Notebook, Pencil...|
|        104|    [Lamp, Pen, Desk]|
|        106| [Tablet, Protecto...|
+-----------+---------------------+

+-----------+--------------------+
|customer_id|collect_set(product)|
+-----------+--------------------+
|        101|[Keyboard, Laptop...|
|        103|[Keyboard, Desk, ...|
|        102|[Mouse, Laptop, D...|
|        105|[Pencil, Notebook...|
|        104|   [Desk, Pen, Lamp]|
|        106|[Notebook, Tablet...|
+-----------+--------------------+



In [31]:
#Take a random sample of 20% of the rows from the DataFrame.

sample_df = practice_df_relprodlist.sample(withReplacement=True, fraction = 0.2, seed = 42)
sample_df.show()

#Perform a stratified sample on the product column, taking 50% of "Laptop" orders and 100% of "Mouse" orders.
strat_sample = practice_df_relprodlist.sampleBy("product", fractions = {"Laptop": 0.5, "Mouse": 1.0}, seed = 123)
strat_sample.show()


+--------+-----------+---------+--------+------+----------+--------------------+----------------+------------------+--------------------+--------------------+--------------------+
|order_id|customer_id|  product|quantity| price|order_date|                tags|         details|          features|  product_attributes|    related_products|Related Product List|
+--------+-----------+---------+--------+------+----------+--------------------+----------------+------------------+--------------------+--------------------+--------------------+
|       6|        104|     Lamp|       2|  35.0|2023-01-17|  furniture,lighting|   BulbType: LED|       id_abc_lamp|                NULL|  [Desk Chair, Desk]|     Desk Chair,Desk|
|       7|        101|   Laptop|       1|1200.0|2023-01-18|  electronics,office|Serial: E5F6G7H8|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|
|      13|        106|Protector|       1|  15.0|2023-01-20|electronics,acces...|    Type: Screen|use

In [38]:
#Find the total quantity of products ordered for each unique tag.

practice_df_relprodlist = practice_df_relprodlist.withColumn("tag_split", split(col("tags"), ","))
tags_df = practice_df_relprodlist.select(col("quantity"), explode(col("tag_split")).alias("alltags"))
tags_df.groupBy("alltags").agg(sum(col("quantity")).alias("total_quantity")).show()

+-----------+--------------+
|    alltags|total_quantity|
+-----------+--------------+
|     office|            36|
|    display|             2|
| stationery|            30|
|   lighting|             3|
|  furniture|             6|
|electronics|            13|
|  accessory|             7|
+-----------+--------------+



In [41]:
'''Create a new column that takes the first 3 characters of the product name and concatenates it with the order_id,
separated by a hyphen (e.g., "Lap-1").'''

substring_df = practice_df_relprodlist.withColumn("short_prod", substring(col("product"),1,3)).withColumn("new_col", concat_ws("-", col("short_prod"), col("order_id")))
substring_df.show()


+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+----------+-------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|           tag_split|short_prod|new_col|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+----------+-------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|[electronics, off...|       Lap|  Lap-1|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M50

In [44]:
#Remove all non-digit characters from the details column.

practice_df_relprodlist.withColumn("no_nondigits", regexp_replace(col("details"), "[^0-9]", "")).show()

#Replace all occurrences of the letter 'e' (case-insensitive) in the product column with the character '@'.

practice_df_relprodlist.withColumn("replace_e", translate(col("product"), "eE", "@")).show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|Related Product List|           tag_split|no_nondigits|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|[Keyboard, Mouse,...|Keyboard,Mouse,Mo...|[electronics, off...|        1234|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{co

In [1]:
#Filter the DataFrame to show only the orders where the product_attributes map contains the key "color" and its value is "silver".

color_df = practice_df_relprodlist.filter(col("product_attributes").getItem("color") == "silver")
color_df.show()


NameError: name 'practice_df_relprodlist' is not defined